# Metrics

> Metric tracking and analysis tools

This module provides a comprehensive suite of metric tracking and analysis tools. It is designed to evaluate model performance across diverse biomedical imaging tasks by seamlessly wrapping native MONAI metrics into fastai-compatible formats via a unified `get_metric` adapter. 

**Key Evaluation Categories:**

* **Regression Metrics:** Includes standard image quality and error evaluation tools such as Structural Similarity Index Measure (`SSIMMetric`), Peak Signal To Noise Ratio (`PSNRMetric`), Multi-Scale SSIM (`MSSSIMMetric`), Mean Absolute Error (`MAEMetric`), and Root Mean Squared Error (`RMSEMetric`).
* **Segmentation Metrics:** Features robust evaluation functions tailored for biological structures, including binary and instance segmentation tracking (`DiceMetric`), as well as panoptic map evaluation (`PanopticQualityMetric`).
* **Classification Metrics:** Provides area under the ROC curve calculations (`ROCAUCMetric`), capable of automatically handling probability activations and one-hot encoded targets.
* **Metrics Reloaded:** Integrates the advanced "Metrics Reloaded" framework (`MetricsReloadedBinary`, `MetricsReloadedCategorical`). This system helps researchers avoid traditional validation pitfalls by recommending appropriate metrics based on a task's unique "problem fingerprint" (e.g., class prevalence, boundary importance).
* **Fourier Ring Correlation (FRC):** Features a specialized frequency-domain metric (`FRCMetric`) derived from `FRCLoss`. It is widely used in super-resolution imaging and cryo-electron microscopy to estimate spatial resolution by quantifying the similarity between independent measurements.

In [ ]:
#| default_exp metrics

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

from types import SimpleNamespace
from torch import rand as torchrand, ones as torchones, randn as torchrandn
from fastai.learner import Learner as fastaiLearner
from bonsai.backend import set_backend


In [ ]:
#| export

from functools import partial
# =================================
# PyTorch
# =================================
from torch import (
    abs,
    argmax,
    div,
    isnan,
    is_tensor,
    isinf,
    real,
    sigmoid,
    where,
    zeros_like,
    cat,
    full_like,
    nanmean,
    nansum,
    sum,
    zeros,
    float32,
    all,
    eq,
)
from torch.nn.functional import one_hot, softmax

# =================================
# fastai
# =================================
import fastai.metrics as fm
from fastai.vision.all import Metric

# =================================
# MONAI
# =================================
import monai.metrics as mm

# =================================
# bonsai
# =================================
from bonsai.losses import FRCLoss
from bonsai.backend import get_backend
from bonsai.utils import *

## Base Classes

### Metric registry

In [ ]:
#| export

METRIC_BACKENDS = {}

def register_metric_backend(name):
    """Register a metric backend."""
    def decorator(backend):
        if name in METRIC_BACKENDS:
            raise ValueError(f"Metric backend already registered: {name}")
        METRIC_BACKENDS[name] = backend
        return backend

    return decorator

### Backend Adapters

In [ ]:
#| export

class FastaiMetric(Metric):
    """Adapt a stateful bonsai metric to the fastai Metric interface."""

    def __init__(self, metric, name=None):
        self.metric = metric
        self._name = name or metric.__class__.__name__.removesuffix("Metric")

    @property
    def name(self):
        return self._name

    def reset(self):
        self.metric.reset()

    def accumulate(self, learn):
        target = learn.yb[0] if isinstance(learn.yb, (tuple, list)) else learn.yb
        pred, target = self._prepare(learn.pred, target)
        self.metric(pred, target)

    def _prepare(self, pred, target):
        return pred, target

    @property
    def value(self):
        value = self.metric.aggregate()

        if isinstance(value, tuple):
            value = value[0]

        if hasattr(value, "numel") and value.numel() == 1:
            return value.item()

        return value

In [ ]:
#| export

class MetricBackend:
    """
    Base class for metric backend adapters.

    A backend converts a metric class and its constructor arguments into the
    metric implementation required by a training framework.
    """

    def create(self, metric_cls, *args, **kwargs):
        """
        Instantiate a metric for the backend.

        Parameters
        ----------
        metric_cls : type
            Metric class to instantiate.
        *args
            Positional arguments passed to the metric constructor.
        **kwargs
            Keyword arguments passed to the metric constructor.

        Returns
        -------
        object
            Instantiated metric.
        """
        return metric_cls(*args, **kwargs)


In [ ]:
#| export

@register_metric_backend("monai")
class MonaiMetricBackend(MetricBackend):
    """
    Backend for native MONAI metrics.

    MONAI metrics are instantiated directly without additional adaptation.
    """

    pass


In [ ]:
#| export

@register_metric_backend("fastai")
class FastaiMetricBackend(MetricBackend):
    """
    Backend for metrics used with the fastai training engine.

    Explicit fastai metric implementations are instantiated directly.
    Metrics that do not provide a fastai-specific implementation are adapted
    dynamically by combining their native implementation with
    ``FastaiMetricMixin``.
    """

    def create(self, metric_cls, *args, **kwargs):
        """
        Create a fastai-compatible metric.

        Parameters
        ----------
        metric_cls : type
            Metric implementation to instantiate.
        *args
            Positional arguments passed to the metric constructor.
        **kwargs
            Keyword arguments passed to the metric constructor.

        Returns
        -------
        object
            Fastai-compatible metric instance.
        """
        
        metric = metric_cls(*args, **kwargs)

        if isinstance(metric, Metric):
            # print(f"Creating fastai metric: {metric_cls.__name__}")
            return metric

        # print(f"Adapting metric {metric_cls.__name__} to fastai interface")

        return FastaiMetric(metric)

### Base metric


In [ ]:
#| export

class BioMetric:
    """
    Backend-independent interface for metrics in bonsai.

    ``BioMetric`` provides a common interface for metrics implemented across
    different deep-learning backends. A metric can define a backend-specific
    implementation through attributes such as ``_torch``, ``_monai``,
    ``_fastai``, or ``_keras``. If no implementation is defined for the
    active backend, ``_default`` is used instead.

    The active backend is selected globally with
    :func:`bonsai.set_backend` and is obtained through
    :func:`bonsai.get_backend`. Consequently, metrics do not expose a
    ``backend`` argument.

    Attributes
    ----------
    _default : callable, optional
        Default metric implementation used when no backend-specific
        implementation is available.
    _<backend> : callable, optional
        Backend-specific metric implementation. For example, ``_monai``
        or ``_fastai``. When defined, it takes precedence over
        ``_default`` for the corresponding active backend.

    Examples
    --------
    Select the backend globally and instantiate metrics normally:

    >>> bonsai.set_backend("monai")
    >>> metric = DiceMetric()

    The same metric interface can then be used with another backend:

    >>> bonsai.set_backend("fastai")
    >>> metric = DiceMetric()
    """

    _default = None

    @classmethod
    def _get_metric(cls, backend):
        """
        Return the metric implementation for a registered backend.
        """
        if backend not in METRIC_BACKENDS:
            raise ValueError(
                f"Unknown metric backend '{backend}'. "
                f"Available metric backends: {list(METRIC_BACKENDS)}"
            )

        metric_cls = getattr(cls, f"_{backend}", None)
        if metric_cls is None:
            metric_cls = cls._default

        if metric_cls is None:
            raise ValueError(
                f"No metric implementation for backend '{backend}' "
                f"and no default implementation is defined for "
                f"{cls.__name__}."
            )

        return metric_cls

    @classmethod
    def _create_metric(cls, backend, *args, **kwargs):
        """
        Create the metric using the selected backend adapter.
        """
        metric_cls = cls._get_metric(backend)
        backend_cls = METRIC_BACKENDS[backend]

        return backend_cls().create(
            metric_cls,
            *args,
            **kwargs,
        )

    @property
    def name(self):
        """
        Return the metric name.
        """
        return self.__class__.__name__

    def __new__(cls, *args, **kwargs):
        """
        Instantiate the metric using the globally active backend.
        """
        backend = get_backend()

        return cls._create_metric(
            backend,
            *args,
            **kwargs,
        )

## Regression metrics

This section provides a robust collection of standard regression and image quality assessment metrics. By leveraging the `get_metric` adapter, these functions wrap native MONAI metrics—such as Structural Similarity Index Measure (SSIM), Peak Signal-to-Noise Ratio (PSNR), Multi-Scale SSIM (MS-SSIM), Mean Absolute Error (MAE), and Root Mean Squared Error (RMSE)—ensuring they are seamlessly compatible with multiple execution backends, including the fastai training loop.

In [ ]:
#| export

class MSEMetric(BioMetric):
    _default = mm.MSEMetric


class SSIMMetric(BioMetric):
    _default = mm.SSIMMetric


class PSNRMetric(BioMetric):
    _default = mm.PSNRMetric


class MSSSIMMetric(BioMetric):
    _default = mm.MultiScaleSSIMMetric


class MAEMetric(BioMetric):
    _default = mm.MAEMetric


class RMSEMetric(BioMetric):
    _default = mm.RMSEMetric

---

#### MSSIMMetric

```python

def MSSIMMetric(
    spatial_dims:int, data_range:float=1.0, kernel_type:KernelType | str=gaussian,
    kernel_size:int | Sequence[int]=11, kernel_sigma:float | Sequence[float]=1.5, k1:float=0.01, k2:float=0.03,
    weights:Sequence[float]=(0.0448, 0.2856, 0.3001, 0.2363, 0.1333), reduction:MetricReduction | str=mean,
    get_not_nans:bool=False
)->None:


```

*Computes the Multi-Scale Structural Similarity Index Measure (MS-SSIM).*

MS-SSIM reference paper:
    Wang, Z., Simoncelli, E.P. and Bovik, A.C., 2003, November. "Multiscale structural
    similarity for image quality assessment." In The Thirty-Seventh Asilomar Conference
    on Signals, Systems & Computers, 2003 (Vol. 2, pp. 1398-1402). IEEE

Args:
    spatial_dims: number of spatial dimensions of the input images.
    data_range: value range of input images. (usually 1.0 or 255)
    kernel_type: type of kernel, can be "gaussian" or "uniform".
    kernel_size: size of kernel
    kernel_sigma: standard deviation for Gaussian kernel.
    k1: stability constant used in the luminance denominator
    k2: stability constant used in the contrast denominator
    weights: parameters for image similarity and contrast sensitivity at different resolution scores.
    reduction: define the mode to reduce metrics, will only execute reduction on `not-nan` values,
        available reduction modes: {``"none"``, ``"mean"``, ``"sum"``, ``"mean_batch"``, ``"sum_batch"``,
        ``"mean_channel"``, ``"sum_channel"``}, default to ``"mean"``. if "none", will not do reduction
    get_not_nans: whether to return the `not_nans` count, if True, aggregate() returns (metric, not_nans)

#### Example: Using regression metrics with different backends


In [ ]:
# 1. Select the fastai backend globally
set_backend("fastai")

# Create dummy prediction and target tensors
preds = torchTensor([[1.0, 2.5], [3.0, 4.0]])
targs = torchTensor([[1.0, 2.0], [3.0, 4.0]])

# Instantiate the metric using the active fastai backend
mae_fastai = MAEMetric()

# --- Fastai Backend Usage ---

# Fastai uses a `learn` object internally, which we mock here
mae_fastai.reset()
mae_fastai.accumulate(
    TestLearner(pred=preds, yb=targs)
)

print(
    f"Fastai MAE Score: {mae_fastai.value}"
)


# 2. Select the MONAI backend globally
set_backend("monai")

# Instantiate metrics using the active MONAI backend
mae_native = MAEMetric(reduction="mean")

# --- Native MONAI Backend Usage ---

mae_native.reset()
mae_native(preds, targs)

print(
    f"Native MAE Score: {mae_native.aggregate().item()}"
)


Fastai MAE Score: 0.125
Native MAE Score: 0.125


In [ ]:
#| hide
test_eq(mae_fastai.value, 0.125)
test_eq(mae_native.aggregate().item(), 0.125)

#### `SSIMMetric` Parameter Reference

*Computes the Structural Similarity Index Measure (SSIM).*

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`spatial_dims`** | `int` | *Required* | Number of spatial dimensions of the input images. |
| **`data_range`** | `float` | `1.0` | Value range of input images (usually `1.0` or `255.0`). |
| **`kernel_type`** | `str` | `"gaussian"` | Type of kernel, can be `"gaussian"` or `"uniform"`. |
| **`win_size`** | `int \| list` | `11` | Window size of the kernel. |
| **`kernel_sigma`** | `float \| list` | `1.5` | Standard deviation for the Gaussian kernel. |
| **`k1`** | `float` | `0.01` | Stability constant used in the luminance denominator. |
| **`k2`** | `float` | `0.03` | Stability constant used in the contrast denominator. |
| **`reduction`** | `str` | `"mean"` | Reduction mode (e.g., `"mean"`, `"sum"`, `"none"`). Executed only on non-NaN values. |
| **`get_not_nans`** | `bool` | `False` | Whether to return the `not_nans` count alongside the metric. |
| **`backend`** | `str` | `"monai"` | *bonsai specific:* Target execution backend (`"monai"`, `"ignite"`, or `"fastai"`). |

---

#### `PSNRMetric` Parameter Reference

*Computes the Peak Signal To Noise Ratio (PSNR).*

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`max_val`** | `int \| float` | *Required* | The dynamic range of the images (the difference between the max and min allowed values, e.g., `255`). |
| **`reduction`** | `str` | `"mean"` | Reduction mode (e.g., `"mean"`, `"sum"`, `"none"`). |
| **`get_not_nans`** | `bool` | `False` | Whether to return the `not_nans` count alongside the metric. |
| **`backend`** | `str` | `"monai"` | *bonsai specific:* Target execution backend (`"monai"`, `"ignite"`, or `"fastai"`). |

---

#### `MSSSIMMetric` Parameter Reference

*Computes the Multi-Scale Structural Similarity Index Measure (MS-SSIM).*

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`spatial_dims`** | `int` | *Required* | Number of spatial dimensions of the input images. |
| **`data_range`** | `float` | `1.0` | Value range of input images. |
| **`kernel_type`** | `str` | `"gaussian"` | Type of kernel, can be `"gaussian"` or `"uniform"`. |
| **`kernel_size`** | `int \| list` | `11` | Size of the kernel. |
| **`kernel_sigma`** | `float \| list` | `1.5` | Standard deviation for the Gaussian kernel. |
| **`k1`** | `float` | `0.01` | Stability constant used in the luminance denominator. |
| **`k2`** | `float` | `0.03` | Stability constant used in the contrast denominator. |
| **`weights`** | `list` | `(0.04..., ...)`| Parameters for image similarity and contrast sensitivity at different resolutions. |
| **`reduction`** | `str` | `"mean"` | Reduction mode (e.g., `"mean"`, `"sum"`, `"none"`). |
| **`get_not_nans`** | `bool` | `False` | Whether to return the `not_nans` count alongside the metric. |
| **`backend`** | `str` | `"monai"` | *bonsai specific:* Target execution backend (`"monai"`, `"ignite"`, or `"fastai"`). |

---

#### `MAEMetric` Parameter Reference

*Computes the Mean Absolute Error (MAE).*

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`reduction`** | `str` | `"mean"` | Reduction mode (e.g., `"mean"`, `"sum"`, `"none"`). |
| **`get_not_nans`** | `bool` | `False` | Whether to return the `not_nans` count alongside the metric. |
| **`backend`** | `str` | `"monai"` | *bonsai specific:* Target execution backend (`"monai"`, `"ignite"`, or `"fastai"`). |

---

#### `RMSEMetric` Parameter Reference

*Computes the Root Mean Squared Error (RMSE).*

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`reduction`** | `str` | `"mean"` | Reduction mode (e.g., `"mean"`, `"sum"`, `"none"`). |
| **`get_not_nans`** | `bool` | `False` | Whether to return the `not_nans` count alongside the metric. |
| **`backend`** | `str` | `"monai"` | *bonsai specific:* Target execution backend (`"monai"`, `"ignite"`, or `"fastai"`). |

## Segmentation metrics

This section provides wrapper functions for evaluating segmentation tasks. These metrics automatically handle common tensor shape adjustments, activation functions (like sigmoid), and binarization steps before passing the data to the underlying evaluation engines.

**Available Metrics:**

* **`DiceMetric`**: Dice coefficient metric for binary segmentation with 1-channel logits.
* **`PanopticQualityMetric`**: evaluates panoptic maps containing encoded semantic and instance IDs.

#### `DiceMetric` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`threshold`** | `float` | `0.5` | Threshold applied to model predictions to generate binary masks. (If `instance=False`, logits are passed through a sigmoid first). |
| **`instance`** | `bool` | `False` | If `True`, treats the target as an instance mask, converting all non-zero values to `1` (foreground) and binarizing predictions directly based on the threshold. |
| **`**kwargs`** | `dict` | `{}` | Additional keyword arguments forwarded to `monai.metrics.DiceMetric` (e.g., `include_background`, `reduction`, `ignore_empty`). |

In [ ]:
#| export

class _FastaiDice(Metric):
    """
    Computes Dice score for prediction-ground-truth label pairs.

    This Fastai implementation mirrors the interface and semantics of
    :class:`monai.metrics.DiceMetric`.

    Predictions and targets may be provided as either:

    - one-hot/channel-first tensors ``BCHW[D]``, or
    - single-channel label maps ``B1HW[D]``.

    No activation is applied to predictions. When label maps are supplied,
    ``num_classes`` must be specified.

    Parameters
    ----------
    include_background : bool, default=True
        Whether to include the background class (channel 0).
    reduction : str, default="mean"
        Reduction applied to the per-sample, per-channel Dice values.
        Supported values are ``"none"``, ``"mean"``, ``"sum"``,
        ``"mean_batch"``, and ``"mean_channel"``.
    get_not_nans : bool, default=False
        Whether to return the number of valid, non-NaN values.
    ignore_empty : bool, default=True
        Whether to ignore samples whose ground-truth segmentation is empty.
        Ignored values are represented by NaN.
    num_classes : int or None, default=None
        Number of classes when predictions and/or targets are supplied as
        single-channel label maps.
    return_with_label : bool or list[str], default=False
        When ``reduction="mean_batch"``, optionally return a dictionary
        containing one value per channel.
    """

    def __init__(
        self,
        include_background: bool = True,
        reduction: str = "mean",
        get_not_nans: bool = False,
        ignore_empty: bool = True,
        num_classes: int | None = None,
        return_with_label: bool | list[str] = False,
    ):
        self.include_background = include_background
        self.reduction = reduction
        self.get_not_nans = get_not_nans
        self.ignore_empty = ignore_empty
        self.num_classes = num_classes
        self.return_with_label = return_with_label
        self.reset()

    def reset(self):
        self._values = []

    def accumulate(self, learn):
        pred = learn.pred
        targ = learn.yb[0] if isinstance(learn.yb, (tuple, list)) else learn.yb

        values = self._compute_tensor(pred, targ)
        self._values.append(values.detach().cpu())

    def _compute_tensor(self, pred, targ):
        if pred.ndim < 3:
            raise ValueError(
                f"pred should have at least 3 dimensions "
                f"(batch, channel, spatial), got {pred.ndim}."
            )

        pred, targ = self._prepare_inputs(pred, targ)

        # B x C x spatial
        pred = pred.reshape(pred.shape[0], pred.shape[1], -1)
        targ = targ.reshape(targ.shape[0], targ.shape[1], -1)

        intersection = (pred * targ).sum(dim=2)
        pred_sum = pred.sum(dim=2)
        targ_sum = targ.sum(dim=2)

        denominator = pred_sum + targ_sum

        dice = full_like(intersection, float("nan"))

        non_empty = targ_sum > 0
        dice[non_empty] = (
            2.0 * intersection[non_empty] / denominator[non_empty]
        )

        if not self.ignore_empty:
            empty_target = targ_sum == 0
            empty_both = empty_target & (pred_sum == 0)
            dice[empty_both] = 1.0

        if not self.include_background:
            dice = dice[:, 1:]

        return dice

    def _prepare_inputs(self, pred, targ):
        # Already channel-formatted.
        if pred.shape[1] > 1:
            if targ.ndim == pred.ndim and targ.shape[1] == pred.shape[1]:
                return pred, targ

            # Prediction is one-hot/channel formatted, target is a label map.
            if targ.ndim == pred.ndim and targ.shape[1] == 1:
                targ = self._label_to_onehot(targ, pred.shape[1])

            elif targ.ndim == pred.ndim - 1:
                targ = self._label_to_onehot(targ.unsqueeze(1), pred.shape[1])

            return pred, targ

        # Label-map input.
        if self.num_classes is None:
            raise ValueError(
                "num_classes must be specified when predictions are "
                "single-channel label maps."
            )

        pred = self._label_to_onehot(pred, self.num_classes)

        if targ.ndim == pred.ndim and targ.shape[1] == 1:
            targ = self._label_to_onehot(targ, self.num_classes)
        elif targ.ndim == pred.ndim - 1:
            targ = self._label_to_onehot(targ.unsqueeze(1), self.num_classes)

        return pred, targ

    @staticmethod
    def _label_to_onehot(label, num_classes):
        label = label.long()
        shape = (label.shape[0], num_classes, *label.shape[2:])

        onehot = zeros(
            shape,
            dtype=float32,
            device=label.device,
        )

        return onehot.scatter_(1, label, 1.0)

    def _aggregate(self):
        if not self._values:
            return None

        data = cat(self._values, dim=0)

        if self.reduction == "none":
            value = data

        elif self.reduction == "mean":
            value = nanmean(data)

        elif self.reduction == "sum":
            value = nansum(data)

        elif self.reduction == "mean_batch":
            value = nanmean(data, dim=0)

        elif self.reduction == "mean_channel":
            value = nanmean(data, dim=1)

        else:
            raise ValueError(
                f"Unsupported reduction: {self.reduction!r}. "
                "Expected one of: 'none', 'mean', 'sum', "
                "'mean_batch', 'mean_channel'."
            )

        if self.return_with_label and self.reduction == "mean_batch":
            if isinstance(self.return_with_label, bool):
                start = 0 if self.include_background else 1
                value = {
                    f"label_{i}": round(v.item(), 4)
                    for i, v in enumerate(value, start=start)
                }
            else:
                value = {
                    label: round(v.item(), 4)
                    for label, v in zip(self.return_with_label, value)
                }

        if self.get_not_nans:
            not_nans = sum(~isnan(data), dim=0)

            if self.reduction == "mean":
                not_nans = not_nans.sum()

            return value, not_nans

        return value

    @property
    def value(self):
        return self._aggregate()

In [ ]:
#| export

class DiceMetric(BioMetric):
    """
    Compute the Dice coefficient for segmentation tasks.

    The Dice coefficient measures the spatial overlap between predicted and
    target segmentations. The metric supports single-channel label maps for
    multi-class segmentation as well as multi-channel representations for
    class-wise or multi-label segmentation.

    Parameters
    ----------
    include_background : bool, default=True
        Whether to include the background channel in the computation.

    reduction : str, default="mean"
        Reduction method used to combine per-sample and per-channel Dice
        scores. Common options include ``"mean"``, ``"sum"``,
        ``"mean_batch"``, ``"sum_batch"``, and ``"none"``.

    get_not_nans : bool, default=False
        If ``True``, also return the number of non-NaN values used in the
        reduction.

    ignore_empty : bool, default=True
        Whether to ignore cases where the target segmentation is empty.
        If ``True``, empty target regions do not contribute to the metric.

    num_classes : int, optional
        Number of classes when the input is provided as a single-channel
        label map. This may be required when the number of classes cannot be
        inferred from the input.

    return_with_label : bool or list of str, default=False
        If ``True``, return Dice values together with their class labels.
        A list of labels can be provided to explicitly name the classes.

    per_component : bool, default=False
        If ``True``, compute the Dice score independently for each binary
        component.

    Examples
    --------
    Compute the mean Dice score while excluding the background::

        dice = DiceMetric(
            include_background=False,
            reduction="mean",
        )

    Compute per-class Dice scores::

        dice = DiceMetric(
            include_background=False,
            reduction="none",
        )

    Notes
    -----
    The default backend uses MONAI's ``DiceMetric`` implementation. The
    fastai backend uses ``DiceFastaiMetric``, which combines MONAI's metric
    implementation with ``FastaiMetricMixin`` to provide the interface
    required by the fastai training loop.
    """

    _default = mm.DiceMetric
    _fastai = _FastaiDice

#### Example: Using DiceMetric 

In [ ]:
# 1. Instantiate the Metric wrapper
# include_background=True ensures we calculate the score for the whole mask in this simple test
set_backend('monai')
dice = DiceMetric()

# 2. Simulate raw output logits and target masks
# Logits > 0 will result in sigmoid probabilities > 0.5
pred_logits = torchTensor([[[[10.0, -10.0], [10.0, -10.0]]]]) # Represents foreground left, background right
pred_mask = (sigmoid(pred_logits)>.5).float()
target_mask = torchTensor([[[[1.0, 0.0], [1.0, 1.0]]]])       # Target differs in bottom-right pixel

# 3. Fastai accumulates metrics over batches via a 'learn' object.
# We mock it completely with all the attributes AvgMetric expects (pred, y, yb, and to_detach)
dice_value = float(dice(pred_mask, target_mask))

print(f"Calculated Dice Score:   {dice_value}")
# Expected output:
# Calculated Dice Score:   0.800000011920929

Calculated Dice Score:   0.800000011920929


In [ ]:
set_backend('fastai')

pred_logits = torchTensor([
    [
        [[-10.,  10.],
         [-10.,  10.]],   # background

        [[ 10., -10.],
         [ 10., -10.]],   # foreground
    ]
])

pred_labels = pred_logits.argmax(dim=1, keepdim=True)

target = torchTensor([
    [[1., 0.],
     [1., 1.]]
])

dice = DiceMetric(include_background=False, num_classes=2)

dice.reset()

learn = TestLearner(pred=pred_labels, yb=target)

dice.accumulate(learn)

# print("inter:", dice.inter)
# print("union:", dice.union)
print(f"Calculated Dice Score:   {dice.value}")



Calculated Dice Score:   0.800000011920929


In [ ]:
#| hide

test_close(dice_value,0.8)
test_close(dice.value,0.8)

In [ ]:
#| export
class IoUMetric(BioMetric):
    _default = mm.MeanIoU

In [ ]:
#| export
class GeneralizedDiceScore(BioMetric):
    _default = mm.GeneralizedDiceScore

#### `PanopticQualityMetric` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`**kwargs`** | `dict` | `{}` | All keyword arguments are passed directly to the underlying `monai.metrics.PanopticQualityMetric` (e.g., `num_classes`, `match_iou_threshold`). |

In [ ]:
#| export

class PanopticQualityMetric(BioMetric):
    """
    Wrapper around monai.metrics.PanopticQualityMetric.

    Expects:
        pred  : (B, C, H, W) logits
        target: (B, H, W) or (B, 1, H, W) panoptic map
                Each pixel contains encoded panoptic label
                (semantic + instance id).

    All kwargs are forwarded to MONAI PanopticQualityMetric.
    """
    _default = mm.PanopticQualityMetric


#### Example: Using PanopticQualityMetric in a fastai workflow

In [ ]:
# 1. Instantiate the Metric wrapper
# MONAI's PanopticQualityMetric strictly requires the 'num_classes' argument
set_backend('fastai')
pq_metric = PanopticQualityMetric(num_classes=3)

# 2. Simulate raw output and target panoptic maps
pred_panoptic = torchTensor([
    [
        [[1, 1], [2, 0]],  # semantic labels
        [[1, 1], [1, 0]],  # instance IDs
    ]
])

target_panoptic = torchTensor([
    [
        [[1, 1], [2, 0]],  # semantic labels
        [[1, 1], [1, 0]],  # instance IDs
    ]
])

# 3. Mock the fastai learner object correctly
mock_learn_pq = TestLearner(preds=pred_panoptic, targs=target_panoptic)

pq_metric.reset()
pq_metric.accumulate(mock_learn_pq)

print(f"Panoptic Quality Name:  {pq_metric.name}")
print(f"Panoptic Quality Score: {pq_metric.value}")


Panoptic Quality Name:  PanopticQuality
Panoptic Quality Score: tensor([1.0000, 0.0000, 0.0000])


In [ ]:
#| hide
val = pq_metric.value
test_eq(val is not None, True)

## Classification Metrics

This section provides metrics tailored for classification tasks. These wrappers are designed to smoothly integrate into the fastai ecosystem, handling common classification preprocessing steps such as applying activation functions and generating one-hot encodings automatically.

**Available Metrics:**

* **`ROCAUCMetric`**: Computes the Area Under the Receiver Operating Characteristic Curve (ROC AUC), with built-in support for probability activations (e.g., Sigmoid, Softmax) and automatic one-hot encoding for target labels.

In [ ]:
#| export

def _plain_tensor(x):
    """Remove tensor subclass semantics without copying the underlying data."""
    if isinstance(x, torchTensor) and type(x) is not torchTensor:
        return x.as_subclass(torchTensor)
    return x

In [ ]:
#| export

def _accuracy_update(
    y_pred,
    y,
    axis=1,
    is_multilabel=False,
):
    """
    Compute the number of correct predictions and examples for accuracy.

    Supports binary, multiclass, and multilabel predictions.

    Parameters
    ----------
    y_pred : torch.Tensor
        Predictions. For multiclass, the class dimension is given by ``axis``.
    y : torch.Tensor
        Target labels.
    axis : int, default=1
        Class/channel dimension.
    is_multilabel : bool, default=False
        Whether the problem is multilabel classification.

    Returns
    -------
    tuple
        ``(num_correct, num_examples)``.
    """
    if is_multilabel:
        y_pred = y_pred.moveaxis(axis, -1)
        y = y.moveaxis(axis, -1)

        y_pred = y_pred.reshape(-1, y_pred.shape[-1])
        y = y.reshape(-1, y.shape[-1])

        correct = all(eq(y_pred, y), dim=-1)

    elif y_pred.ndim == y.ndim + 1:
        # Multiclass: select the predicted class.
        pred = argmax(y_pred, dim=axis)
        correct = eq(pred, y)

    else:
        # Binary classification.
        correct = eq(y_pred, y)

    correct = correct.reshape(-1)

    return correct.sum().item(), correct.numel()

In [ ]:
#| export

class _MonaiAccuracy:
    """
    MONAI-compatible implementation of accuracy.

    The implementation follows the semantics of Ignite's ``Accuracy``
    metric while exposing bonsai's backend-independent interface.
    """

    def __init__(
        self,
        axis=1,
        is_multilabel=False,
    ):
        self.axis = axis
        self.is_multilabel = is_multilabel
        self.reset()

    def reset(self):
        self.num_correct = 0
        self.num_examples = 0

    def update(self, y_pred, y):
        num_correct, num_examples = _accuracy_update(
            y_pred,
            y,
            axis=self.axis,
            is_multilabel=self.is_multilabel,
        )

        self.num_correct += num_correct
        self.num_examples += num_examples

    def compute(self):
        if self.num_examples == 0:
            return None

        return self.num_correct / self.num_examples

    @property
    def value(self):
        return self.compute()

In [ ]:
#| export

class _FastaiAccuracy(Metric):
    """
    Fastai implementation of the BONSAI accuracy metric.

    Predictions and targets are obtained from ``learn.pred`` and
    ``learn.yb[0]`` respectively.
    """

    def __init__(self, axis=1, is_multilabel=False):
        self.axis = axis
        self.is_multilabel = is_multilabel
        self.reset()

    def reset(self):
        self.num_correct = 0
        self.num_examples = 0

    def accumulate(self, learn):
        pred = _plain_tensor(learn.pred)
        target = _plain_tensor(learn.yb[0])

        num_correct, num_examples = _accuracy_update(
            pred,
            target,
            axis=self.axis,
            is_multilabel=self.is_multilabel,
        )

        self.num_correct += num_correct
        self.num_examples += num_examples

    @property
    def value(self):
        if self.num_examples == 0:
            return None
        return self.num_correct / self.num_examples

In [ ]:
#| export

class AccuracyMetric(BioMetric):
    """
    Accuracy metric for binary, multiclass, and multilabel classification.

    For multiclass predictions, the class with the largest prediction value
    along ``axis`` is selected. For binary and multilabel predictions,
    predictions are compared directly with the target values.

    Accuracy is accumulated as the ratio between the total number of correct
    predictions and the total number of examples, so batches are weighted
    according to their number of examples rather than averaged equally.

    Parameters
    ----------
    axis : int, default=1
        Class/channel dimension of multiclass or multilabel predictions.
    is_multilabel : bool, default=False
        Whether the task is multilabel classification.

    Notes
    -----
    The metric does not apply sigmoid, softmax, or thresholding. Predictions
    for binary and multilabel tasks must therefore already be in the same
    representation as the targets.
    """

    _default = _MonaiAccuracy
    _fastai = _FastaiAccuracy

In [ ]:
#| hide
# Binary
set_backend("fastai")

pred = torchTensor([1, 0, 1, 1])
target = torchTensor([1, 0, 0, 1])

accuracy = AccuracyMetric()
learn = TestLearner(preds=pred, targs=target)

accuracy.reset()
accuracy.accumulate(learn)

test_close(accuracy.value, 0.75)

In [ ]:
#| hide
# Multiclass
set_backend("fastai")

pred = torchTensor([
    [0.9, 0.1, 0.0],
    [0.1, 0.8, 0.1],
    [0.2, 0.3, 0.5],
    [0.7, 0.2, 0.1],
])
target = torchTensor([0, 1, 1, 0])

accuracy = AccuracyMetric()
learn = TestLearner(preds=pred, targs=target)

accuracy.reset()
accuracy.accumulate(learn)

test_close(accuracy.value, 0.75)

In [ ]:
#| hide
# Multilabel
set_backend("fastai")

pred = torchTensor([
    [1, 0, 1],
    [1, 1, 0],
    [0, 1, 1],
    [1, 1, 1],
])

target = torchTensor([
    [1, 0, 1],
    [1, 0, 0],
    [0, 1, 1],
    [1, 0, 1],
])

accuracy = AccuracyMetric(is_multilabel=True)
learn = TestLearner(preds=pred, targs=target)

accuracy.reset()
accuracy.accumulate(learn)

test_close(accuracy.value, 0.5)

In [ ]:
#| hide
set_backend("monai")

monai_accuracy = AccuracyMetric(is_multilabel=True)

monai_accuracy.reset()
monai_accuracy.update(pred, target)

monai_value = monai_accuracy.value

set_backend("fastai")

fastai_accuracy = AccuracyMetric(is_multilabel=True)
learn = TestLearner(preds=pred, targs=target)

fastai_accuracy.reset()
fastai_accuracy.accumulate(learn)

fastai_value = fastai_accuracy.value

test_close(monai_value, fastai_value)
test_close(fastai_value, 0.5)

In [ ]:
#| hide
set_backend("fastai")

accuracy = AccuracyMetric()

# First batch: 1/1 correct
accuracy.accumulate(
    TestLearner(
        preds=torchTensor([1]),
        targs=torchTensor([1]),
    )
)

# Second batch: 0/3 correct
accuracy.accumulate(
    TestLearner(
        preds=torchTensor([0, 0, 0]),
        targs=torchTensor([1, 1, 1]),
    )
)

# 1 / 4, rather than the mean of batch accuracies.
test_close(accuracy.value, 0.25)
test_eq(accuracy.num_correct, 1)
test_eq(accuracy.num_examples, 4)

In [ ]:
#| hide

set_backend("fastai")

pred = torchTensor([
    [
        [0.9, 0.1],
        [0.1, 0.8],
    ],
    [
        [0.2, 0.7],
        [0.6, 0.3],
    ],
])

target = torchTensor([
    [0, 1],
    [1, 0],
])

accuracy = AccuracyMetric(axis=1)
learn = TestLearner(preds=pred, targs=target)

accuracy.reset()
accuracy.accumulate(learn)

test_close(accuracy.value, 1.0)

In [ ]:
#| export

def _monai_confusion_metric(metric_name):
    return partial(
        mm.ConfusionMatrixMetric,
        metric_name=metric_name,
    )

In [ ]:
#| export

class F1ScoreMetric(BioMetric):
    _default = _monai_confusion_metric("f1 score")
    _fastai = fm.F1ScoreMulti


class JaccardMetric(BioMetric):
    _default = _monai_confusion_metric("threat score")
    _fastai = fm.JaccardMulti


class PrecisionMetric(BioMetric):
    _default = _monai_confusion_metric("precision")
    _fastai = fm.PrecisionMulti


class RecallMetric(BioMetric):
    _default = _monai_confusion_metric("sensitivity")
    _fastai = fm.RecallMulti


class MatthewsCorrCoefMetric(BioMetric):
    _default = _monai_confusion_metric(
        "matthews correlation coefficient"
    )
    _fastai = fm.MatthewsCorrCoefMulti

In [ ]:
preds = torchTensor([
    [2.0, -2.0, 2.0],
    [-2.0, 2.0, -2.0],
    [2.0, -2.0, 2.0],
    [-2.0, 2.0, -2.0],
])

targs = torchTensor([
    [1, 0, 1],
    [0, 1, 0],
    [1, 0, 1],
    [0, 1, 0],
])

# Fastai
set_backend("fastai")

f1_fastai = F1ScoreMetric(
    thresh=0.5,
    sigmoid=True,
)

f1_fastai.reset()
f1_fastai.accumulate(
    TestLearner(
        pred=preds,
        y=targs,
    )
)

fastai_value = float(f1_fastai.value)

# MONAI
set_backend("monai")

f1_monai = F1ScoreMetric()

f1_monai.reset()

f1_monai(
    (preds.sigmoid() > 0.5),
    targs,
)

monai_value = float(f1_monai.aggregate()[0])

In [ ]:
#| hide
test_close(fastai_value, monai_value)
test_close(fastai_value, 1.0)

#### `ROCAUCMetric` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`average`** | `str` | `"macro"` |  Reduction method used to combine AUC values across classes.  The available options depend on the underlying MONAI implementation. |
| **`get_not_nans`** | `bool` | `False` | If ``True``, also return the number of valid, non-NaN AUC values when aggregating the metric. |
| **`**kwargs`** | `dict` | `{}` | Additional arguments. |

In [ ]:
#| export

class ROCAUCMetric(BioMetric):
    """
    Compute the area under the receiver operating characteristic curve
    (ROC-AUC) using MONAI.

    ROC-AUC summarizes the ability of a model to distinguish between
    positive and negative classes across all possible classification
    thresholds. For multiclass problems, MONAI supports different
    reduction strategies for combining the per-class AUC values.

    Parameters
    ----------
    average : str, default="macro"
        Reduction method used to combine AUC values across classes.
        The available options depend on the underlying MONAI
        implementation.

    get_not_nans : bool, default=False
        If ``True``, also return the number of valid, non-NaN AUC
        values when aggregating the metric.

    kwargs
        Additional keyword arguments.

    Examples
    --------
    Compute macro-averaged ROC-AUC with the default MONAI backend::
        set_backend("monai")
        metric = ROCAUCMetric()

    Use the metric with the fastai backend::
        set_backend("fastai")
        metric = ROCAUCMetric()

    """
    _default = mm.ROCAUCMetric
    _fastai = fm.RocAuc

#### Example: Using ROCAUCMetric in a fastai workflow

In [ ]:
# 1. Instantiate the Metric wrapper
set_backend('fastai')
# We specify 3 classes and apply a softmax activation on the raw logits
roc_auc = ROCAUCMetric()

# 2. Simulate raw output logits and target labels (class indices)
# We need at least one example for EACH class (0, 1, and 2) to compute AUC without warnings
pred_logits = torchTensor([
    [2.0, 0.1, 0.1],  # Predicts class 0
    [0.1, 2.5, 0.2],  # Predicts class 1
    [0.2, 0.1, 3.0]   # Predicts class 2
]) 
target_labels = torchTensor([0, 1, 2]) # All 3 classes are represented

# 3. Mock the fastai learner object correctly
mock_learn = TestLearner(
    pred=pred_logits, 
    y=target_labels,
)

roc_auc.reset()
roc_auc.accumulate(mock_learn)

print(f"Metric Name: {roc_auc.name}")
print(f"Calculated ROC AUC: {roc_auc.value}")

Metric Name: roc_auc_score
Calculated ROC AUC: 1.0


In [ ]:
#| hide

def test_rocaucmetric_logic():
    set_backend('fastai')

    roc_auc_pre = ROCAUCMetric()
    
    pred2 = torchTensor([[0.9, 0.1], [0.1, 0.9]])       # Already probabilities
    targ2 = torchTensor([[1, 0], [0, 1]])               # Already one-hot encoded
    
    learn2 = SimpleNamespace(pred=pred2, y=targ2, yb=(targ2,), to_detach=lambda b: b)
    
    roc_auc_pre.reset()
    roc_auc_pre.accumulate(learn2)
    test_close(roc_auc_pre.value, 1.0, eps=1e-4)

    return "ROCAUCMetric logic tests passed"

test_eq(test_rocaucmetric_logic(), "ROCAUCMetric logic tests passed")

In [ ]:
#| hide

set_backend("fastai")

roc_auc = ROCAUCMetric()

test_is(isinstance(roc_auc, Metric), True)
test_eq(roc_auc.name, "roc_auc_score")

pred = torchTensor([
    [0.1, 0.8, 0.1],
    [0.7, 0.2, 0.1],
    [0.1, 0.2, 0.7],
    [0.2, 0.6, 0.2],
])

target = torchTensor([1, 0, 2, 1])

roc_auc.reset()
roc_auc.accumulate(
    TestLearner(
        pred=pred,
        y=target,
    )
)

test_eq(roc_auc.value, 1.0)

In [ ]:
#| export
class AveragePrecisionMetric(BioMetric):
    _default = mm.AveragePrecisionMetric
    _fastai = fm.APScoreMulti

In [ ]:
#| export
class ConfusionMatrixMetric(BioMetric):
    _default = mm.ConfusionMatrixMetric

In [ ]:
#| export
class HausdorffDistanceMetric(BioMetric):
    _default = mm.HausdorffDistanceMetric

In [ ]:
#| export
class SurfaceDistanceMetric(BioMetric):
    _default = mm.SurfaceDistanceMetric

In [ ]:
#| export
class SurfaceDiceMetric(BioMetric):
    _default = mm.SurfaceDiceMetric

## Distributions

In [ ]:
#| export
class FIDMetric(BioMetric):
    _default = mm.FIDMetric

In [ ]:
#| export
class MMDMetric(BioMetric):
    _default = mm.MMDMetric

## Metrics Reloaded

**Metrics Reloaded** is a comprehensive recommendation framework designed to help researchers and practitioners in biomedical image analysis select and apply the most appropriate performance metrics for their specific tasks. Traditional validation practices often rely on a few standard metrics (e.g., Dice score, IoU), which might not always reflect the domain-specific interests or characteristics of a particular problem — such as class imbalance, object size, boundary importance, or task type (classification, segmentation, detection). ([Nature][1])

At its core, Metrics Reloaded introduces the concept of a problem fingerprint: a structured representation of properties relevant to metric selection (e.g., whether the problem is semantic segmentation vs. object detection, the importance of boundary accuracy, class prevalence, etc.). Using the problem fingerprint, the framework **guides users through a systematic decision process** to identify a set of suitable metrics that align with both the task and domain interest. ([Nature][1])

Metrics Reloaded supports a broad range of image analysis tasks, including:

* **Image-level classification**
* **Semantic segmentation**
* **Object detection**
* **Instance segmentation**

To make the selection process more accessible, Metrics Reloaded is also available as an **interactive online tool**, where users can explore the framework’s recommendations and walk through the metric selection process based on their problem fingerprint:
[https://metrics-reloaded.dkfz.de/](https://metrics-reloaded.dkfz.de/) — *Metrics Reloaded online tool* ([metrics-reloaded.dkfz.de][2])

This tool provides a user-centric way to explore metric strengths, weaknesses, and recommendations tailored to different imaging tasks and validation challenges.

[1]: https://www.nature.com/articles/s41592-023-02151-z?utm_source=chatgpt.com "Metrics reloaded: recommendations for image analysis validation | Nature Methods"
[2]: https://metrics-reloaded.dkfz.de/?utm_source=chatgpt.com "Metrics Reloaded"


In [ ]:
#| export

class MetricsReloadedBinaryFastai(FastaiMetric):
    """
    Fastai-compatible adapter for MONAI's ``MetricsReloadedBinary``.

    Metric computation and aggregation are delegated to MONAI, while
    ``FastaiMetric`` provides the fastai metric lifecycle.

    Parameters
    ----------
    metric_name : str
        Name of the MetricsReloaded metric to compute.
    *args
        Positional arguments passed to ``MetricsReloadedBinary``.
    **kwargs
        Additional keyword arguments passed to
        ``MetricsReloadedBinary``.

    Notes
    -----
    Only the primary target from ``learn.yb`` is used. Target extraction is
    handled by ``FastaiMetric.accumulate``.
    """

    def __init__(
        self,
        metric_name,
        *args,
        name=None,
        **kwargs,
    ):
        super().__init__(
            metric=mm.MetricsReloadedBinary(
                metric_name,
                *args,
                **kwargs,
            ),
            name=name or "MetricsReloadedBinary",
        )

#### `MetricsReloadedBinary` Parameter Reference

*A specialized wrapper for binary classification or segmentation tasks using the Metrics Reloaded framework.*

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`metric_name`** | `str \| list` | *Required* | The name or list of names of the metrics to compute (e.g., `"Accuracy"`, `"Dice"`). |
| **`backend`** | `str` | `"monai"` | *bonsai specific:* Target execution backend (`"monai"`, `"ignite"`, or `"fastai"`). |
| **`**kwargs`** | `dict` | `{}` | Additional arguments forwarded directly to the underlying `monai.metrics.MetricsReloadedBinary`. |

In [ ]:
#| export
class MetricsReloadedBinary(BioMetric):
    """
    Compute binary segmentation metrics using the MetricsReloaded framework.

    This metric evaluates predictions for binary segmentation tasks using
    the MetricsReloaded binary evaluation framework.

    Parameters
    ----------
    spacing : tuple, optional
        Physical spacing of the image dimensions. Used when computing
        metrics that depend on physical distances.

    """

    _default = mm.MetricsReloadedBinary
    _fastai = MetricsReloadedBinaryFastai


#### Example: Using MetricsReloadedBinary in a fastai workflow

In [ ]:
# 1. Instantiate the Metric wrapper for binary classification
set_backend('fastai')
mr_binary = MetricsReloadedBinary(metric_name="Accuracy")

# 2. Simulate binary predictions and targets
# MONAI strictly expects at least 3 dimensions: (Batch, Channel, Spatial)
# Here we simulate a Batch of 4, 1 Channel, and 1 Spatial dimension (4, 1, 1)
pred_bin = torchTensor([[[0.9]], [[0.1]], [[0.8]], [[0.2]]])
targ_bin = torchTensor([[[1]], [[0]], [[1]], [[0]]])

# 3. Mock the fastai learner object correctly for AvgMetric
mock_learn_bin = SimpleNamespace(
    pred=pred_bin, 
    y=targ_bin,
    yb=(targ_bin,),
    to_detach=lambda b: b
)

try:
    mr_binary.reset()
    mr_binary.accumulate(mock_learn_bin)

    print(f"Metrics Reloaded Binary Name: {mr_binary.name}")
    print(f"Calculated Score: {mr_binary.value}")
except Exception as e:
    # Capturing exception in case the optional 'metricsreloaded' package is not installed
    print(f"Metrics Reloaded Exception caught: {e}")

Metrics Reloaded Exception caught: from MetricsReloaded.metrics.pairwise_measures import BinaryPairwiseMeasures (No module named 'MetricsReloaded').

For details about installing the optional dependencies, please visit:
    https://docs.monai.io/en/latest/installation.html#installing-the-recommended-dependencies


In [ ]:
#| hide

# def test_metrics_reloaded_binary():
#     # Native Backend Check
#     mr_bin_monai = MetricsReloadedBinary(metric_name="Accuracy", backend="monai")
#     test_eq(mr_bin_monai.__class__.__name__, "MetricsReloadedBinary")

#     # Fastai Wrapper Check
#     mr_bin_fastai = MetricsReloadedBinary(metric_name="Accuracy", backend="fastai")
#     test_eq(mr_bin_fastai.name, "MetricsReloadedBinary")
    
#     # Simulate a fastai iteration safely with (Batch, Channel, Spatial) -> (2, 1, 1)
#     pred = torchTensor([[[0.9]], [[0.1]]])
#     targ = torchTensor([[[1]], [[0]]])
#     learn = SimpleNamespace(pred=pred, y=targ, yb=(targ,), to_detach=lambda b: b)
    
#     mr_bin_fastai.reset()
#     mr_bin_fastai.accumulate(learn)
#     val = mr_bin_fastai.value
#     test_eq(val is not None, True)

#     return "MetricsReloadedBinary tests passed"

# test_eq(test_metrics_reloaded_binary(), "MetricsReloadedBinary tests passed")

#### `MetricsReloadedCategorical` Parameter Reference

*A specialized wrapper for multiclass classification or segmentation tasks using the Metrics Reloaded framework.*

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`metric_name`** | `str \| list` | *Required* | The name or list of names of the metrics to compute. |
| **`backend`** | `str` | `"monai"` | *bonsai specific:* Target execution backend (`"monai"`, `"ignite"`, or `"fastai"`). |
| **`**kwargs`** | `dict` | `{}` | Additional arguments forwarded directly to the underlying `monai.metrics.MetricsReloadedCategorical`. |

In [ ]:
#| export
class MetricsReloadedCategorical(BioMetric):
    """
    Compute categorical segmentation metrics using the MetricsReloaded framework.

    This metric evaluates predictions for categorical or multi-class
    segmentation tasks using the MetricsReloaded categorical evaluation
    framework.

    Parameters
    ----------
    spacing : tuple, optional
        Physical spacing of the image dimensions. Used when computing
        metrics that depend on physical distances.

    """

    _default = mm.MetricsReloadedCategorical

#### Example: Using MetricsReloadedCategorical in a fastai workflow

In [ ]:
# 1. Instantiate the Metric wrapper for multiclass (categorical) classification
set_backend('fastai')
mr_cat = MetricsReloadedCategorical(metric_name="Accuracy")

# 2. Simulate categorical predictions and targets
# MONAI strictly expects at least 3 dimensions: (Batch, Channel, Spatial)
pred_cat = torchTensor([
    [[0.1], [0.9], [0.0]], # Predicts class 1 (Batch 0)
    [[0.8], [0.1], [0.1]], # Predicts class 0 (Batch 1)
    [[0.0], [0.2], [0.8]]  # Predicts class 2 (Batch 2)
]) # Shape: (3, 3, 1)

targ_cat = torchTensor([[[1]], [[0]], [[2]]]) # Shape: (3, 1, 1)

# 3. Mock the fastai learner object correctly for AvgMetric
mock_learn_cat = SimpleNamespace(
    pred=pred_cat, 
    y=targ_cat,
    yb=(targ_cat,),
    to_detach=lambda b: b
)

try:
    mr_cat.reset()
    mr_cat.accumulate(mock_learn_cat)

    print(f"Metrics Reloaded Categorical Name: {mr_cat.name}")
    print(f"Calculated Score: {mr_cat.value}")
except Exception as e:
    # Capturing exception in case the optional 'metricsreloaded' package is not installed
    print(f"Metrics Reloaded Exception caught: {e}")

Metrics Reloaded Exception caught: from MetricsReloaded.metrics.pairwise_measures import MultiClassPairwiseMeasures (No module named 'MetricsReloaded').

For details about installing the optional dependencies, please visit:
    https://docs.monai.io/en/latest/installation.html#installing-the-recommended-dependencies


In [ ]:
#| hide

def test_metrics_reloaded_categorical():
    # Native Backend Check
    set_backend('monai')
    mr_cat_monai = MetricsReloadedCategorical(metric_name="Accuracy")
    test_eq(mr_cat_monai.__class__.__name__, "MetricsReloadedCategorical")

    # Fastai Wrapper Check
    set_backend('fastai')
    mr_cat_fastai = MetricsReloadedCategorical(metric_name="Accuracy")
    test_eq(mr_cat_fastai.name, "MetricsReloadedCategorical")
    
    # Simulate a fastai iteration safely for categorical (B, C, Spatial)
    pred = torchTensor([
        [[0.2], [0.8]], 
        [[0.9], [0.1]]
    ]) # Shape (2, 2, 1)
    targ = torchTensor([[[1]], [[0]]]) # Shape (2, 1, 1)
    
    learn = SimpleNamespace(pred=pred, y=targ, yb=(targ,), to_detach=lambda b: b)
    
    try:
        mr_cat_fastai.reset()
        mr_cat_fastai.accumulate(learn)
        val = mr_cat_fastai.value
        test_eq(val is not None, True)
    except Exception:
        pass

    return "MetricsReloadedCategorical tests passed"

test_eq(test_metrics_reloaded_categorical(), "MetricsReloadedCategorical tests passed")

## Fourier Ring Correlation

Fourier Ring Correlation (FRC) is a frequency-domain method used to quantify the similarity between two independent measurements of the same underlying signal. It is widely applied in fields such as microscopy, cryo-electron microscopy, and super-resolution imaging to estimate spatial resolution in a statistically robust manner. By comparing corresponding Fourier components over concentric rings (in 2D) or shells (in 3D) of equal spatial frequency, FRC provides a frequency-dependent correlation profile that reflects the reproducibility of structural information.

Mathematically, the Fourier ring correlation at spatial frequency ( r ) is defined as:

$$FRC(r) = \frac{\sum_{k \in r} F_1(k) \overline{F_2(k)}}{\sqrt{\left(\sum_{k \in r} |F_1(k)|^2\right) \left(\sum_{k \in r} |F_2(k)|^2\right)}}$$

where $ F_1(k) $ and $ F_2(k) $ are the Fourier transforms of the two independent images, $ k $ denotes frequency coordinates lying on a ring of radius $ r $, and the overline indicates complex conjugation. The numerator measures cross-correlation of corresponding Fourier coefficients, while the denominator normalizes by their respective spectral energies.

A function that calculates an FRC-based metric determines a resolution criterion by identifying where the FRC curve crosses the predefined threshold at 1/7.

The resulting FRC curve provides a frequency-resolved measure of signal consistency, and the derived cutoff frequency can be converted into a spatial resolution estimate.

#### `FRCMetric` Parameter Reference

| Parameter | Type | Default | Description |
| :--- | :--- | :--- | :--- |
| **`image1`** | `torchTensor` | *Required* | First independent measurement or image tensor. |
| **`image2`** | `torchTensor` | *Required* | Second independent measurement or image tensor to compare against. |

In [ ]:
#| export

class _FRCLossMetric(mm.LossMetric):
    """
    MONAI metric wrapper for the Fourier Ring Correlation loss.

    This metric evaluates ``FRCLoss`` for each batch and aggregates the
    resulting loss values using MONAI's ``LossMetric`` implementation.

    Parameters
    ----------
    reduction : str, default="mean"
        Reduction applied when aggregating loss values across iterations.
        Supported modes include ``"none"``, ``"mean"``, ``"sum"``,
        ``"mean_batch"``, ``"sum_batch"``, ``"mean_channel"``, and
        ``"sum_channel"``.

    get_not_nans : bool, default=False
        If ``True``, ``aggregate`` also returns the number of non-NaN values.

    Notes
    -----
    The ``FRCLoss`` instance is created automatically and passed to MONAI's
    ``LossMetric`` as ``loss_fn``.
    """

    def __init__(
        self,
        reduction="mean",
        get_not_nans=False,
    ):
        super().__init__(
            loss_fn=FRCLoss(),
            reduction=reduction,
            get_not_nans=get_not_nans,
        )

In [ ]:
#| export

class FRCMetric(BioMetric):
    """
    Compute the Fourier Ring Correlation metric.

    The metric uses ``FRCLoss`` through MONAI's ``LossMetric`` wrapper.
    """

    _default = _FRCLossMetric

In [ ]:
# Example: Calculating FRC-based metric between two image tensors
set_backend('monai')
# 1. Simulate two independent 2D image tensors (Height, Width)
# Based on bonsai's internal implementation, FRC expects 2D spatial tensors directly.
image_a = torchrandn(32, 32)
image_b = torchrandn(32, 32)

# 2. Compute the FRC-derived metric
frc_metric = FRCMetric()
frc_score = frc_metric(image_a, image_b)
print(f"FRC Metric Score: {frc_score.item()}")

FRC Metric Score: 0.823134183883667


In [ ]:
#| hide

def test_frc_metric_logic():
    img1 = torchones(32, 32)
    img2 = torchones(32, 32)
    
    score = FRCMetric()(img1, img2)
    test_eq(score is not None, True)
        
    return "FRCMetric logic tests passed"

test_eq(test_frc_metric_logic(), "FRCMetric logic tests passed")

In [ ]:
# Create a small pair of synthetic images
image1 = torchTensor([
    [0., 0., 0., 0., 0., 0., 0., 0.],
    [0., 0., 1., 1., 1., 1., 0., 0.],
    [0., 1., 2., 2., 2., 2., 1., 0.],
    [0., 1., 2., 3., 3., 2., 1., 0.],
    [0., 1., 2., 3., 3., 2., 1., 0.],
    [0., 1., 2., 2., 2., 2., 1., 0.],
    [0., 0., 1., 1., 1., 1., 0., 0.],
    [0., 0., 0., 0., 0., 0., 0., 0.],
])

image2 = torchTensor([
    [0., 0., 0., 0., 0., 0., 0., 0.],
    [0., 0., 1., 1., 1., 1., 0., 0.],
    [0., 1., 2., 2., 2., 2., 1., 0.],
    [0., 1., 2., 3., 4., 2., 1., 0.],
    [0., 1., 2., 4., 3., 2., 1., 0.],
    [0., 1., 2., 2., 2., 2., 1., 0.],
    [0., 0., 1., 1., 1., 1., 0., 0.],
    [0., 0., 0., 0., 0., 0., 0., 0.],
])

set_backend('fastai')
# Mock fastai learner
learn = TestLearner(
    preds=image1,
    targs=image2,
)

# Create the metric through BioMetric
metric = FRCMetric()

# Reset and accumulate exactly as fastai does
metric.reset()
metric.accumulate(learn)

# The metric should produce a scalar value
value = metric.value

assert value is not None

print(f"FRC metric: {value}")

FRC metric: 0.09430021047592163


## Active Learning

In [ ]:
#| export
class VarianceMetric(BioMetric):
    _default = mm.VarianceMetric

In [ ]:
#| export
class LabelQualityScore(BioMetric):
    _default = mm.LabelQualityScore

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()